# TieBreaker — ModernBERT-large cross-encoder tie 분류기

In [1]:
import os, csv, gc, json, random, math, time, shutil, datetime
import warnings; warnings.filterwarnings('ignore')
from transformers import logging as hf_logging; hf_logging.set_verbosity_error()
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'

import numpy as np
import pandas as pd
import inspect
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoConfig, AutoModel, TrainingArguments, Trainer, TrainerCallback,
                          EarlyStoppingCallback)
from transformers.trainer_pt_utils import LengthGroupedSampler
from transformers.trainer_utils import get_last_checkpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss

In [2]:
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
def set_seeds(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seeds(SEED)
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print('device', DEVICE, '| bf16', USE_BF16, '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

device cuda | bf16 True | NVIDIA L40S


In [3]:
# ── 데이터 + 역할 분리 (stage2 = npz valid+holdout, 나머지 = train_enc) ──
WORK_DIR = '/home/ubuntu/workspace'          # AWS 작업 루트 — 모든 경로의 기준 (읽기/쓰기 일치)
DATA_DIR = f'{WORK_DIR}/origin'
REF_NPZ  = f'{WORK_DIR}/outputs/logits_export.npz'
assert os.path.exists(os.path.join(DATA_DIR, 'train.csv')) and os.path.exists(REF_NPZ), '경로 확인'

ref = np.load(REF_NPZ)
STAGE2_IDS = np.concatenate([ref['valid_ids'], ref['hold_ids']])
stage2_lab = np.concatenate([ref['valid_labels'], ref['hold_labels']]).astype(np.int64)

df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
df['label'] = np.select([df['winner_model_a'] == 1, df['winner_model_b'] == 1, df['winner_tie'] == 1], [0, 1, 2], -1)
assert (df['label'] != -1).all()
df['is_stage2'] = df['id'].isin(set(STAGE2_IDS.tolist()))
assert int(df['is_stage2'].sum()) == len(STAGE2_IDS)

enc_df = df[~df['is_stage2']].reset_index(drop=True)
s2_df  = df.set_index('id').loc[STAGE2_IDS].reset_index()
assert (np.select([s2_df['winner_model_a']==1, s2_df['winner_model_b']==1, s2_df['winner_tie']==1],[0,1,2],-1) == stage2_lab).all()
y_s2 = (stage2_lab == 2).astype(np.int64)

enc_df['is_tie'] = (enc_df['label'] == 2).astype(int)
tr_df, va_df = train_test_split(enc_df, test_size=0.10, stratify=enc_df['is_tie'], random_state=SEED, shuffle=True)
tr_df = tr_df.reset_index(drop=True); va_df = va_df.reset_index(drop=True)
print(f'train_enc {len(enc_df):,} (tie {enc_df.is_tie.mean()*100:.1f}%) -> fit {len(tr_df):,} / val {len(va_df):,}'
      f' | stage2 {len(s2_df):,} (tie {y_s2.mean()*100:.1f}%)')

train_enc 46,729 (tie 30.9%) -> fit 42,056 / val 4,673 | stage2 10,748 (tie 30.9%)


In [4]:
# ── 토크나이저 + budget 배분 (A/B 반반+slack, 앞6뒤4). [BR] 미사용(개행 네이티브 보존) ──
MODEL_NAME = 'answerdotai/ModernBERT-large'
REVISION   = '45bb4654a4d5aaff24dd11d4781fa46d39bf8c13'   # pinned commit of the official repository / 공식 저장소의 고정 커밋
ATTN       = 'sdpa'            # flash-attn 설치 시 'flash_attention_2'로 (2048 학습 가속)
MAX_LEN    = 4096              # Q|A|B 절단 0.6%(실측). 길이정렬 배칭이라 짧은 배치는 4096 패딩 안 함
                              #   + ModernBERT local/global 교대 어텐션이라 장문도 저렴 -> 4096 감당 가능.
                              #   VRAM 게이트는 worst-batch 프로브. (타협 필요 시 3072=1.6% / 2048=4.2%)
Q_FLOOR    = 256               # 긴 Q는 바닥 256, 응답이 안 쓴 예산은 Q가 회수
HEAD_FRAC  = 0.6
CHAR_CAP   = 20000             # 토크나이즈 전 char head-tail 컷(속도) — 토큰 budget보다 넉넉

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=REVISION)
CLS, SEP, PAD = tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id
BUDGET = MAX_LEN - 4           # [CLS] Q [SEP] A [SEP] B [SEP]
assert None not in (CLS, SEP, PAD), 'CLS/SEP/PAD 확인'
print(f'{MODEL_NAME} | CLS {CLS} SEP {SEP} PAD {PAD} | budget {BUDGET} | 개행 네이티브 보존([BR] 미사용)')

def parse_text(x):
    if pd.isna(x): return ''
    try: turns = json.loads(x)
    except (json.JSONDecodeError, TypeError): return str(x)
    return ' '.join(t for t in turns if isinstance(t, str)) if isinstance(turns, list) else str(turns)

def char_head_tail(s):
    if len(s) <= CHAR_CAP: return s
    h = int(CHAR_CAP * HEAD_FRAC); return s[:h] + s[-(CHAR_CAP - h):]

def head_tail(ids, k):
    if len(ids) <= k: return list(ids)
    h = int(k * HEAD_FRAC); t = k - h
    return list(ids[:h]) + list(ids[-t:]) if t > 0 else list(ids[:h])

def alloc_qab(lq, la, lb):                 # Q 바닥 256 -> 나머지 A/B 반반+slack -> 잔여는 Q가 회수
    kq = min(lq, Q_FLOOR)
    rem = BUDGET - kq
    half = rem // 2
    if la <= half and lb <= half: ka, kb = la, lb
    elif la <= half: ka = la; kb = min(lb, rem - la)
    elif lb <= half: kb = lb; ka = min(la, rem - lb)
    else: ka, kb = half, rem - half
    kq = min(lq, kq + (BUDGET - kq - ka - kb))          # Train_run alloc_q의 slack 회수
    return kq, ka, kb

def build_triple(iq, ia, ib):              # 수동 구성 (ModernBERT build_inputs는 특수토큰 미삽입)
    kq, ka, kb = alloc_qab(len(iq), len(ia), len(ib))
    return ([CLS] + head_tail(iq, kq) + [SEP]
            + head_tail(ia, ka) + [SEP] + head_tail(ib, kb) + [SEP])

def tok_texts(series):                     # 개행 그대로 토크나이즈 (to_model_text 없음)
    return tokenizer([char_head_tail(parse_text(t)) for t in series], add_special_tokens=False)['input_ids']

answerdotai/ModernBERT-large | CLS 50281 SEP 50282 PAD 50283 | budget 4092 | 개행 네이티브 보존([BR] 미사용)


In [5]:
# ── Dataset(단일 시퀀스, swap 증강 binary) + collate. TTA는 tie_probs 인자 교환(A↔B) ──
class TieDataset(Dataset):
    def __init__(self, frame, has_label=True, swap_aug=False):
        frame = frame.reset_index(drop=True)
        self.has_label, self.swap_aug = has_label, swap_aug
        self.Q = tok_texts(frame['prompt'])
        self.A = tok_texts(frame['response_a'])
        self.B = tok_texts(frame['response_b'])
        self.lengths = [min(len(q), Q_FLOOR) + min(len(a), BUDGET//2) + min(len(b), BUDGET//2)
                        for q, a, b in zip(self.Q, self.A, self.B)]   # swap 불변 근사
        self.labels = (frame['label'] == 2).astype(int).tolist() if has_label else None

    def __len__(self): return len(self.A)

    def __getitem__(self, i):
        a, b = self.A[i], self.B[i]
        if self.swap_aug and random.random() < 0.5:    # A/B만 교환, Q 고정 — tie 라벨 불변
            a, b = b, a
        item = {'seq': build_triple(self.Q[i], a, b)}
        if self.has_label: item['labels'] = self.labels[i]
        return item

def collate_fn(batch):
    seqs = [b['seq'] for b in batch]
    L = max(len(s) for s in seqs)
    ids  = np.full((len(seqs), L), PAD, np.int64); mask = np.zeros((len(seqs), L), np.int64)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = s; mask[i, :len(s)] = 1
    out = {'input_ids': torch.from_numpy(ids), 'attention_mask': torch.from_numpy(mask)}
    if 'labels' in batch[0]:
        out['labels'] = torch.tensor([b['labels'] for b in batch], dtype=torch.long)
    return out

In [6]:
# ── cross-encoder: 1회 인코딩 + AttnPool(N(0,0.02)) + binary 헤드 (DeBERTa 때와 동일 풀링) ──
class AttnPool(nn.Module):                          # Train_run과 동일 — N(0,0.02) 초기화
    def __init__(self, h):
        super().__init__()
        self.score = nn.Linear(h, 1)
        nn.init.normal_(self.score.weight, std=0.02) # 0 근처 -> 초기 near-uniform(≈mean), 1스텝부터 gradient
        nn.init.zeros_(self.score.bias)
    def forward(self, last_hidden, mask):
        s = self.score(last_hidden).squeeze(-1)
        s = s.float().masked_fill(mask == 0, float('-inf'))
        w = torch.softmax(s, dim=1).unsqueeze(-1)
        return (last_hidden.float() * w).sum(dim=1)

class TieBreakerModel(nn.Module):
    def __init__(self, model_name, num_classes=2, attn='sdpa'):
        super().__init__()
        # reference_compile=False: 길이정렬 배칭은 배치마다 길이가 달라 torch.compile이 매번 재컴파일 -> 극심한 지연.
        #   이 버전은 from_pretrained kwarg로 주면 모델 __init__에 그대로 넘어가 TypeError -> config에 설정해 넘긴다.
        _cfg = AutoConfig.from_pretrained(model_name, revision=REVISION)
        if hasattr(_cfg, 'reference_compile'):
            _cfg.reference_compile = False
        self.encoder = AutoModel.from_pretrained(model_name, revision=REVISION, config=_cfg, attn_implementation=attn)
        h = self.encoder.config.hidden_size
        self.pool = AttnPool(h)                     # ModernBERT는 mean 풀링 계열 -> AttnPool은 그 일반화(학습형 가중)
        self.head = nn.Sequential(nn.Dropout(0.1), nn.Linear(h, h), nn.GELU(),
                                  nn.Dropout(0.1), nn.Linear(h, num_classes))
        self.loss_fct = nn.CrossEntropyLoss()

    def gradient_checkpointing_enable(self, **kw):
        self.encoder.gradient_checkpointing_enable(**kw)

    def forward(self, input_ids, attention_mask, labels=None):
        with torch.autocast(device_type='cuda', dtype=AMP_DTYPE, enabled=torch.cuda.is_available()):
            hs = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
            pooled = self.pool(hs, attention_mask)  # 마스크 attention 풀링 (CLS/mean 대신)
            logits = self.head(pooled)
        logits = logits.float()
        if labels is None: return {'logits': logits}
        return {'loss': self.loss_fct(logits, labels), 'logits': logits}

In [7]:
# ── 하이퍼파라미터 + TrainingArguments (선택 지표 = eval_logloss) ──
LR          = 1e-5           # large는 낮게
EPOCHS      = 3
WARMUP_RATIO= 0.1
WEIGHT_DECAY= 0.01
GRAD_CLIP   = 1.0
BATCH_SIZE  = 8             # @4096: A40 48G 8 안전 / A10 24G면 4로↓ & GRAD_ACCUM 4↑ (worst-batch 프로브가 잡음)
GRAD_ACCUM  = 2            # eff batch = 16 (BATCH 낮추면 GA 올려 유지)
EVAL_STEPS  = 500
EARLY_STOP_PATIENCE = 5    # eval_logloss가 이 횟수만큼 개선 없으면 조기 종료
BEST_K      = 5            # BestSaver: 점수 상위 K개 체크포인트 별도 보관 (앙상블 재료)
OUTPUT_DIR  = f'{WORK_DIR}/outputs/tiebreaker'

steps_per_epoch = math.ceil(len(tr_df) / (BATCH_SIZE * GRAD_ACCUM))
print(f'steps/epoch {steps_per_epoch} · total {steps_per_epoch*EPOCHS} | eval every {EVAL_STEPS}')

training_args = TrainingArguments(
    output_dir=f'{OUTPUT_DIR}/run',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=32,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR, weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO, max_grad_norm=GRAD_CLIP,
    eval_strategy='steps', eval_steps=EVAL_STEPS,
    save_strategy='steps', save_steps=EVAL_STEPS, save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model='eval_logloss', greater_is_better=False,
    bf16=USE_BF16, fp16=not USE_BF16 and torch.cuda.is_available(),
    gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
    dataloader_num_workers=4, eval_accumulation_steps=10, logging_steps=50,
    report_to='none', disable_tqdm=False, remove_unused_columns=False, seed=SEED)

steps/epoch 2629 · total 7887 | eval every 500


In [8]:
# ── compute_metrics(binary) + CSV 로거 + Stage2Reporter(보고 전용, TTA) ──
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    p = torch.softmax(torch.tensor(np.array(logits), dtype=torch.float32), 1).numpy()[:, 1]
    return {'auc': roc_auc_score(labels, p), 'logloss': log_loss(labels, np.c_[1 - p, p], labels=[0, 1])}

@torch.no_grad()
def tie_probs(model, Q_ids, A_ids, B_ids):          # 단일 뷰 P(tie), 길이정렬 배칭
    was_train = model.training; model.eval()
    n = len(A_ids); out = np.zeros(n, np.float32)
    order = np.argsort([len(Q_ids[i]) + len(A_ids[i]) + len(B_ids[i]) for i in range(n)]); bs = 32
    for st in range(0, n, bs):
        idx = order[st:st + bs]
        seqs = [build_triple(Q_ids[j], A_ids[j], B_ids[j]) for j in idx]
        L = max(len(s) for s in seqs)
        ids = np.full((len(seqs), L), PAD, np.int64); m = np.zeros((len(seqs), L), np.int64)
        for r, s in enumerate(seqs): ids[r, :len(s)] = s; m[r, :len(s)] = 1
        logits = model(input_ids=torch.from_numpy(ids).to(DEVICE),
                       attention_mask=torch.from_numpy(m).to(DEVICE))['logits']
        out[idx] = torch.softmax(logits, 1)[:, 1].float().cpu().numpy()
    if was_train: model.train()
    return out

def tie_tta(model, Q_ids, A_ids, B_ids):            # TTA = A/B 교환 (Q 고정)
    return (tie_probs(model, Q_ids, A_ids, B_ids) + tie_probs(model, Q_ids, B_ids, A_ids)) / 2

class TrainingLogger(TrainerCallback):
    FIELDS = ['step', 'epoch', 'loss', 'learning_rate', 'eval_loss', 'eval_auc', 'eval_logloss',
              'stage2_auc', 'stage2_ll']
    def __init__(self, path): self.path = path; self._seen = set(); self.last_s2 = ''; self.last_s2_ll = ''
    def on_train_begin(self, args, state, control, **kw):
        if state.is_world_process_zero:
            os.makedirs(os.path.dirname(self.path), exist_ok=True)
            if not os.path.exists(self.path) or os.path.getsize(self.path) == 0:
                with open(self.path, 'a', newline='') as f:
                    csv.DictWriter(f, fieldnames=self.FIELDS, extrasaction='ignore').writeheader()
    def on_log(self, args, state, control, logs=None, **kw):
        if not (state.is_world_process_zero and logs): return
        kind = 'eval' if 'eval_loss' in logs else 'train' if 'loss' in logs else None
        if kind is None or (state.global_step, kind) in self._seen: return
        self._seen.add((state.global_step, kind))
        row = {k: logs.get(k, '') for k in self.FIELDS}
        row['step'] = state.global_step; row['epoch'] = logs.get('epoch', state.epoch)
        if kind == 'eval': row['stage2_auc'] = self.last_s2; row['stage2_ll'] = self.last_s2_ll
        with open(self.path, 'a', newline='') as f:
            csv.DictWriter(f, fieldnames=self.FIELDS, extrasaction='ignore').writerow(row)

class Stage2Reporter(TrainerCallback):              # stage2 = 보고 전용 (선택 절대 금지)
    def __init__(self, Q_ids, A_ids, B_ids, y, logger):
        self.Q, self.A, self.B, self.y, self.logger = Q_ids, A_ids, B_ids, y, logger
    def on_evaluate(self, args, state, control, model=None, **kw):
        if not state.is_world_process_zero: return
        prob = tie_tta(model, self.Q, self.A, self.B)
        auc = roc_auc_score(self.y, prob)
        ll = log_loss(self.y, np.clip(prob, 1e-6, 1 - 1e-6))
        self.logger.last_s2 = round(float(auc), 4); self.logger.last_s2_ll = round(float(ll), 4)
        print(f'  [stage2 report-only] step {state.global_step}: TTA AUC {auc:.4f} / ll {ll:.4f}  (선택엔 미사용)')

# BestSaver — save_total_limit(최신순)이 지우는 '중간 구간 상위 체크포인트'를 점수 상위 K개로 별도 보관.
#   가중치만 복사(optimizer/scheduler/rng 제외), 재개 시 랭킹 복구. Train_run과 동일 (metric은 generic).
class BestSaver(TrainerCallback):
    def __init__(self, k=5, output_dir=None, metric='eval_auc', greater_is_better=True):
        self.k = k; self.metric = metric
        self.sign = -1 if greater_is_better else 1      # sign*score 오름차순 = 좋은 것 먼저
        self.skip = shutil.ignore_patterns('optimizer.pt', 'scheduler.pt', 'rng_state*.pth', 'scaler.pt')
        self.best_dir = os.path.join(output_dir, 'best_checkpoints')
        self.state_file = os.path.join(self.best_dir, 'best_k_state.json')
        self.best_checkpoints = []
        os.makedirs(self.best_dir, exist_ok=True); self._load_state()
    def _load_state(self):
        if os.path.exists(self.state_file):
            try:
                self.best_checkpoints = json.load(open(self.state_file, encoding='utf-8'))
                print(f'[BestSaver] state restored ({len(self.best_checkpoints)} entries)')
            except Exception as e:
                print(f'[BestSaver] state load failed, fresh: {e}')
    def _save_state(self):
        for rank, e in enumerate(self.best_checkpoints, 1): e['rank'] = rank
        try: json.dump(self.best_checkpoints, open(self.state_file, 'w', encoding='utf-8'), indent=4, ensure_ascii=False)
        except Exception as e: print(f'[BestSaver] state save failed: {e}')
    def on_save(self, args, state, control, **kwargs):
        if not state.is_world_process_zero: return
        src = os.path.join(args.output_dir, f'checkpoint-{state.global_step}')
        if not os.path.isdir(src): raise FileNotFoundError(f'[BestSaver] not found: {src}')
        score = None
        for log in reversed(state.log_history):
            if log.get('step') == state.global_step and self.metric in log: score = log[self.metric]; break
        if score is None:
            print(f'[BestSaver] no {self.metric} at step {state.global_step}, skipped'); return
        entry = {'score': score, 'metric': self.metric, 'step': state.global_step,
                 'epoch': round(state.epoch, 4) if state.epoch else 0,
                 'saved_at': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
        self.best_checkpoints.append(entry)
        self.best_checkpoints.sort(key=lambda x: self.sign * x['score']); self._save_state()
        if any(e['step'] == state.global_step for e in self.best_checkpoints[:self.k]):
            dest = os.path.join(self.best_dir, f'checkpoint-{state.global_step}')
            if not os.path.exists(dest):
                print(f'[BestSaver] rank {self.best_checkpoints.index(entry)+1} ({self.metric}={score:.4f}) -> {dest}')
                for i in range(3):
                    try: shutil.copytree(src, dest, ignore=self.skip); break
                    except Exception:
                        shutil.rmtree(dest, ignore_errors=True)
                        if i == 2: raise
                        time.sleep(3)
        if len(self.best_checkpoints) > self.k:
            dropped = self.best_checkpoints[self.k:]; self.best_checkpoints = self.best_checkpoints[:self.k]; self._save_state()
            for e in dropped:
                p = os.path.join(self.best_dir, f'checkpoint-{e["step"]}')
                if os.path.exists(p): print(f'[BestSaver] drop {e["score"]:.4f} @ {e["step"]}'); shutil.rmtree(p, ignore_errors=True)

In [9]:
# ── 길이그룹 배칭 커스텀 트레이너 (Train_run과 동일) ──
class GroupedTrainer(Trainer):
    def _get_train_sampler(self, train_dataset=None):
        ds = train_dataset if train_dataset is not None else self.train_dataset
        return LengthGroupedSampler(
            batch_size=self.args.train_batch_size * self.args.gradient_accumulation_steps,
            dataset=ds, lengths=ds.lengths, model_input_name=None)
    def _get_eval_sampler(self, eval_dataset):
        return np.argsort(np.asarray(eval_dataset.lengths)).tolist()

for name in ['_get_train_sampler', '_get_eval_sampler']:
    assert list(inspect.signature(getattr(Trainer, name)).parameters) == \
           list(inspect.signature(getattr(GroupedTrainer, name)).parameters), f'{name} 시그니처 불일치'
print('*** LSB activated ***')

*** LSB activated ***


In [10]:
# ── 데이터셋·모델·트레이너 + 스모크(불변식 + worst-batch OOM 프로브) ──
train_ds = TieDataset(tr_df, swap_aug=True)
valid_ds = TieDataset(va_df)
Q_s2 = tok_texts(s2_df['prompt'])
A_s2, B_s2 = tok_texts(s2_df['response_a']), tok_texts(s2_df['response_b'])
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1) 불변식 (모델 불필요): [CLS]시작 + [SEP]3 + 길이<=MAX_LEN
for i in range(min(2000, len(train_ds))):
    s = train_ds[i]['seq']
    assert s[0] == CLS and s.count(SEP) == 3 and len(s) <= MAX_LEN
print('전처리 불변식 통과')

# 2) worst-batch OOM 프로브 — '버릴 모델'로 (본 학습 모델을 프로브가 오염시키지 않게, Train_run과 동일)
_idx = np.argsort(train_ds.lengths)[-(BATCH_SIZE * GRAD_ACCUM):]
print('probe batch lengths:', sorted(train_ds.lengths[i] for i in _idx))
class _Probe(Dataset):
    def __init__(s, base, idx): s.base, s.idx = base, list(idx)
    def __len__(s): return len(s.idx)
    def __getitem__(s, i): return s.base[s.idx[i]]
    @property
    def lengths(s): return [s.base.lengths[i] for i in s.idx]
_pm = TieBreakerModel(MODEL_NAME, attn=ATTN).to(DEVICE)   # 프로브 전용, 즉시 폐기
_pa = TrainingArguments(output_dir=f'{OUTPUT_DIR}/_probe', max_steps=1,
    per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR, max_grad_norm=GRAD_CLIP, bf16=USE_BF16, fp16=not USE_BF16 and torch.cuda.is_available(),
    gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
    dataloader_num_workers=2, report_to='none', remove_unused_columns=False, seed=SEED)
torch.cuda.reset_peak_memory_stats()
try:
    GroupedTrainer(model=_pm, args=_pa, train_dataset=_Probe(train_ds, _idx), data_collator=collate_fn).train()
    print(f'worst-batch 프로브 통과 — peak VRAM {torch.cuda.max_memory_allocated()/2**30:.1f} GiB')
finally:
    del _pm; gc.collect(); torch.cuda.empty_cache()

# 3) 본 학습 모델·트레이너 — 프로브와 분리된 새 모델 (오염 없음)
model = TieBreakerModel(MODEL_NAME, attn=ATTN).to(DEVICE)
print(f'params {sum(p.numel() for p in model.parameters())/1e6:.0f}M | hidden {model.encoder.config.hidden_size} | pool AttnPool')
logger = TrainingLogger(f'{OUTPUT_DIR}/train_log.csv')
trainer = GroupedTrainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=valid_ds,
                         data_collator=collate_fn, compute_metrics=compute_metrics,
                         callbacks=[logger, Stage2Reporter(Q_s2, A_s2, B_s2, y_s2, logger),
                                    BestSaver(k=BEST_K, output_dir=OUTPUT_DIR, metric='eval_logloss', greater_is_better=False),
                                    EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)])
print('스모크 통과 — trainer.train() 안전')

전처리 불변식 통과
probe batch lengths: [4348, 4348, 4348, 4348, 4348, 4348, 4348, 4348, 4348, 4348, 4348, 4348, 4348, 4348, 4348, 4348]


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

{'train_runtime': '42.54', 'train_samples_per_second': '0.376', 'train_steps_per_second': '0.024', 'train_loss': '0.6797', 'epoch': '1'}
worst-batch 프로브 통과 — peak VRAM 8.6 GiB


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

params 396M | hidden 1024 | pool AttnPool
스모크 통과 — trainer.train() 안전


In [11]:
# ── 학습(resume-aware) → best 모델(val logloss 기준)에서 stage2 예측 저장 ──
last_ckpt = get_last_checkpoint(training_args.output_dir) if os.path.isdir(training_args.output_dir) else None
print('resume from:', last_ckpt or 'scratch')
trainer.train(resume_from_checkpoint=last_ckpt)     # load_best_model_at_end -> model = val-best

p_tie = tie_tta(model, Q_s2, A_s2, B_s2).astype(np.float32)   # stage2 최종 예측(TTA), 선택 무접촉이라 clean
auc = roc_auc_score(y_s2, p_tie); ll = log_loss(y_s2, np.clip(p_tie, 1e-6, 1 - 1e-6))
print('\n===== tie 판별력 (stage2 AUC) =====')
print(f'  {"인코더(평균)":<22} 0.6250')
print(f'  {"U-LGBM-swap(산술)":<22} 0.6452')
print(f'  {"TieBreaker(ModernBERT-L)":<22} {auc:.4f}  <-- 본대')
print(f'\nstage2: AUC {auc:.4f} | tie ll {ll:.4f} | 현 최고(0.645) 대비 {auc-0.6452:+.4f}')

PARQ = f'{WORK_DIR}/outputs/tiebreaker_stage2.parquet'
pd.DataFrame({'id': STAGE2_IDS, 'p_tie_ce': p_tie}).to_parquet(PARQ)

# ── 모델 저장 (Train_run과 동일 구조) ──
# BestSaver 사본(top-5)은 가중치뿐 — 구조·토크나이저는 전부 동일하니 폴더 루트에 공용 1벌만 저장.
#   커스텀 nn.Module이라 Trainer가 config를 안 남기므로, 이게 없으면 사본을 되살릴 수 없다.
BEST_DIR = f'{OUTPUT_DIR}/best_checkpoints'
if os.path.isdir(BEST_DIR):
    model.encoder.config.save_pretrained(BEST_DIR); tokenizer.save_pretrained(BEST_DIR)
    print('best_checkpoints(top-5):', sorted(d for d in os.listdir(BEST_DIR) if d.startswith('checkpoint')))
# Trainer 보관 best 1개 — config/tokenizer/예측 번들
BEST_CKPT = trainer.state.best_model_checkpoint
if BEST_CKPT and os.path.isdir(BEST_CKPT):
    model.encoder.config.save_pretrained(BEST_CKPT); tokenizer.save_pretrained(BEST_CKPT)
    shutil.copy2(PARQ, os.path.join(BEST_CKPT, 'tiebreaker_stage2.parquet'))
    print('best ckpt:', BEST_CKPT, '|', sorted(os.listdir(BEST_CKPT)))
trainer.save_model(f'{OUTPUT_DIR}/best'); tokenizer.save_pretrained(f'{OUTPUT_DIR}/best')
print(f'saved: {PARQ} / {OUTPUT_DIR}/best/ / {BEST_DIR}/ (top-{BEST_K})')

resume from: scratch


Step,Training Loss,Validation Loss,Auc,Logloss
500,0.614245,0.614301,0.565213,0.614301
1000,0.601680,0.617642,0.567592,0.617642
1500,0.624470,0.616618,0.549760,0.616618
2000,0.607083,0.613235,0.566585,0.613235
2500,0.635225,0.611275,0.565446,0.611275
3000,0.618645,0.609503,0.581179,0.609503
3500,0.622947,0.611380,0.579452,0.611380
4000,0.611596,0.611131,0.571501,0.611131
4500,0.622090,0.615959,0.584287,0.615959
5000,0.606258,0.609464,0.582215,0.609464


  [stage2 report-only] step 500: TTA AUC 0.5815 / ll 0.6130  (선택엔 미사용)


[BestSaver] rank 1 (eval_logloss=0.6143) -> /home/ubuntu/workspace/outputs/tiebreaker/best_checkpoints/checkpoint-500


  [stage2 report-only] step 1000: TTA AUC 0.5807 / ll 0.6156  (선택엔 미사용)


...

[BestSaver] rank 2 (eval_logloss=0.6095) -> /home/ubuntu/workspace/outputs/tiebreaker/best_checkpoints/checkpoint-7000


[BestSaver] drop 0.6113 @ 2500


  [stage2 report-only] step 7500: TTA AUC 0.6005 / ll 0.6061  (선택엔 미사용)



===== tie 판별력 (stage2 AUC) =====
  인코더(평균)                0.6250
  U-LGBM-swap(산술)        0.6452
  TieBreaker(ModernBERT-L) 0.5983  <-- 본대

stage2: AUC 0.5983 | tie ll 0.6060 | 현 최고(0.645) 대비 -0.0469
best_checkpoints(top-5): ['checkpoint-3000', 'checkpoint-4000', 'checkpoint-5000', 'checkpoint-6500', 'checkpoint-7000']
best ckpt: /home/ubuntu/workspace/outputs/tiebreaker/run/checkpoint-5000 | ['config.json', 'model.safetensors', 'optimizer.pt', 'rng_state.pth', 'scheduler.pt', 'tiebreaker_stage2.parquet', 'tokenizer.json', 'tokenizer_config.json', 'trainer_state.json', 'training_args.bin']


saved: /home/ubuntu/workspace/outputs/tiebreaker_stage2.parquet / /home/ubuntu/workspace/outputs/tiebreaker/best/ / /home/ubuntu/workspace/outputs/tiebreaker/best_checkpoints/ (top-5)
